# 03 — MCP: RAG + agentes conectados por el Model Context Protocol

Tercer notebook de la serie incremental — aquí **se junta todo**:

```
 notebook 01                    notebook 03                     notebook 02
┌────────────────────┐      ┌──────────────────────┐      ┌────────────────────┐
│ RAG multimodal     │      │  servidor MCP        │      │ agente LangGraph   │
│ Qdrant + Gemini    │ ◀────│  (FastMCP, stdio)    │◀─MCP─│ (cliente MCP)      │
│ + búsqueda híbrida │      │  herramientas:       │      │ LLM: Ollama cloud  │
└────────────────────┘      │  buscar / detalle    │      └────────────────────┘
                            └──────────────────────┘
```

## ¿Por qué un protocolo?

En el notebook 02, las herramientas viven **dentro** del proceso del agente:
funciones de Python enlazadas con `bind_tools`. Funciona, pero no escala
organizacionalmente: si mañana quieres esas mismas herramientas en Claude
Desktop, en Claude Code, en otro agente de otro equipo… reescribes la
integración cada vez ($M$ hosts × $N$ herramientas = $M{\times}N$ puentes).

El **Model Context Protocol (MCP)** estandariza ese enchufe: cada herramienta
se implementa **una vez** como *servidor MCP*, y cualquier *host* compatible
la consume ($M + N$ piezas). Conceptos:

- **Servidor MCP** — expone *tools* (funciones invocables), *resources*
  (datos leíbles por URI) y *prompts* (plantillas). El nuestro envolverá el
  RAG de TecnoMarket.
- **Host / cliente MCP** — la app donde vive el LLM (Claude Desktop, un agente
  LangGraph…); descubre las capacidades del servidor en tiempo de ejecución.
- **Transporte** — `stdio` (el host lanza el servidor como subproceso; ideal
  local) o HTTP (*streamable HTTP*; ideal remoto). Usaremos `stdio`.

> **Prerrequisitos:** notebook 01 ejecutado (colección en Qdrant), mismo
> `.env`. El servidor MCP reutiliza la API de Gemini para embeber consultas.

In [ ]:
# Carga la configuración desde module4-genai/.env (cópiala de .env.example).
import os
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
if not (ROOT / "rag").exists():          # si ejecutas desde notebooks/
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2")
EMBEDDING_DIM = int(os.getenv("EMBEDDING_DIM", "768"))
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "https://ollama.com")
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
COLLECTION = os.getenv("QDRANT_COLLECTION", "tecnomarket")

print("Módulo:", ROOT)
print("Embeddings:", GEMINI_EMBEDDING_MODEL, f"({EMBEDDING_DIM} dim)")
print("Qdrant:", QDRANT_URL, "| colección:", COLLECTION)
print("GEMINI_API_KEY definida:", bool(os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")))
print("OLLAMA_API_KEY definida:", bool(os.getenv("OLLAMA_API_KEY")))

## 1. El servidor MCP (FastMCP)

`FastMCP` (del SDK oficial `mcp`) convierte funciones de Python en un servidor
MCP con decoradores — la misma filosofía de `@tool` de LangChain, pero
sirviendo por protocolo en lugar de enlazar en proceso.

Escribimos el servidor a un archivo (`mcp/tecnomarket_server.py`) porque un
servidor stdio es un **proceso independiente**: el cliente lo lanzará con
`python tecnomarket_server.py`. Fíjate en que:

- expone **2 tools** (`buscar_catalogo`, `detalle_producto`) que replican las
  herramientas del notebook 02, ahora desacopladas del agente;
- expone **1 resource** (`politica://{nombre}`) — los documentos de políticas
  como datos direccionables por URI, sin pasar por búsqueda;
- carga el `.env` del módulo y habla con Qdrant + Gemini igual que el
  notebook 01 (búsqueda densa, para mantener el servidor corto).

In [ ]:
%%writefile ../mcp/tecnomarket_server.py
"""Servidor MCP de TecnoMarket — expone el RAG del Módulo 4 por protocolo.

Generado desde notebooks/03_mcp_rag_agentes.ipynb. Ejecutar directo:
    uv run python mcp/tecnomarket_server.py     (transporte stdio)
"""
import json
import os
from pathlib import Path

from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

MODULO = Path(__file__).resolve().parent.parent
load_dotenv(MODULO / ".env")

from google import genai                      # noqa: E402
from google.genai import types as gtypes     # noqa: E402
from qdrant_client import QdrantClient, models  # noqa: E402

GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2")
EMBEDDING_DIM = int(os.getenv("EMBEDDING_DIM", "768"))
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")
COLLECTION = os.getenv("QDRANT_COLLECTION", "tecnomarket")

mcp = FastMCP("tecnomarket")
_gclient = genai.Client()
_qdrant = QdrantClient(url=QDRANT_URL)


def _embed_consulta(texto: str) -> list[float]:
    r = _gclient.models.embed_content(
        model=GEMINI_EMBEDDING_MODEL,
        contents=f"task: search result | query: {texto}",
        config=gtypes.EmbedContentConfig(output_dimensionality=EMBEDDING_DIM),
    )
    return list(r.embeddings[0].values)


@mcp.tool()
def buscar_catalogo(consulta: str, top_k: int = 4) -> str:
    """Busca en la base de conocimiento de TecnoMarket (políticas de envíos,
    devoluciones y garantías, y catálogo de productos) los pasajes más
    relevantes para una consulta en lenguaje natural."""
    hits = _qdrant.query_points(
        collection_name=COLLECTION, query=_embed_consulta(consulta),
        limit=top_k, with_payload=True,
    ).points
    bloques = []
    for i, h in enumerate(hits, 1):
        p = h.payload
        if p["tipo"] == "producto_imagen":
            bloques.append(f"[{i}] foto del producto {p['sku']} — {p['nombre']}")
        else:
            origen = p.get("fuente") or f"{p['sku']} — {p['nombre']}"
            bloques.append(f"[{i}] (fuente: {origen})\n{p['texto']}")
    return "\n\n".join(bloques) if bloques else "Sin resultados."


@mcp.tool()
def detalle_producto(sku: str) -> str:
    """Devuelve la ficha exacta de un producto de TecnoMarket dado su SKU
    (formato TM-XXXX)."""
    hits, _ = _qdrant.scroll(
        collection_name=COLLECTION, limit=1, with_payload=True,
        scroll_filter=models.Filter(must=[
            models.FieldCondition(key="sku",
                                  match=models.MatchValue(value=sku.strip().upper())),
            models.FieldCondition(key="tipo",
                                  match=models.MatchValue(value="producto_texto")),
        ]),
    )
    if not hits:
        return f"No existe el SKU {sku}."
    p = hits[0].payload
    return json.dumps({"sku": p["sku"], "nombre": p["nombre"],
                       "descripcion": p["texto"]}, ensure_ascii=False)


@mcp.resource("politica://{nombre}")
def politica(nombre: str) -> str:
    """Texto completo de una política de TecnoMarket: envios, devoluciones o
    garantias."""
    ruta = MODULO / "rag" / "catalog" / "docs" / f"{nombre}.md"
    if not ruta.exists():
        return f"No existe la política '{nombre}'."
    return ruta.read_text(encoding="utf-8")


if __name__ == "__main__":
    mcp.run(transport="stdio")

## 2. Hablar con el servidor "a mano" (cliente MCP puro)

Antes de conectarlo a un agente, inspeccionamos el servidor con el cliente del
SDK, para ver el protocolo desnudo:

1. lanzar el servidor como subproceso (stdio),
2. `initialize` (negociación de capacidades),
3. `list_tools` — **descubrimiento**: el cliente no sabe de antemano qué hay,
4. `call_tool` y `read_resource`.

El SDK de MCP es **asíncrono**; en Jupyter podemos usar `await` directamente
en la celda (el notebook ya corre un event loop).

In [ ]:
import sys

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVIDOR = StdioServerParameters(
    command=sys.executable,
    args=[str(ROOT / "mcp" / "tecnomarket_server.py")],
)

async with stdio_client(SERVIDOR) as (lectura, escritura):
    async with ClientSession(lectura, escritura) as sesion:
        await sesion.initialize()

        tools = await sesion.list_tools()
        print("Tools descubiertas:")
        for t in tools.tools:
            print(f"  - {t.name}: {t.description.splitlines()[0]}")

        r = await sesion.call_tool("buscar_catalogo",
                                   {"consulta": "¿cuánto cuesta el envío express?",
                                    "top_k": 2})
        print("\ncall_tool('buscar_catalogo') →\n", r.content[0].text[:400])

        recurso = await sesion.read_resource("politica://garantias")
        print("\nread_resource('politica://garantias') →",
              recurso.contents[0].text[:120], "…")

## 3. El agente del notebook 02, ahora con herramientas MCP

`langchain-mcp-adapters` traduce las tools MCP descubiertas a herramientas de
LangChain. El agente es el mismo de antes (LangGraph + LLM en Ollama cloud) —
pero ya **no importa las funciones**: las descubre por protocolo. Si mañana el
servidor agrega una herramienta, el agente la ve sin cambiar una línea.

Las herramientas MCP son asíncronas, así que usamos `ainvoke`/`astream`.

In [ ]:
# ⚙️ El modelo de Ollama cloud se elige AQUÍ, en el notebook.
#    Catálogo: https://ollama.com/search?c=cloud . Algunas opciones:
#      "minimax-m3:cloud"    razonador (thinking)
#      "kimi-k3:cloud"       multimodal (visión); se factura como "extra usage"
#      "gpt-oss:120b-cloud"  incluido en el plan gratuito
OLLAMA_MODEL = "minimax-m3:cloud"

# Si el modelo elegido no está disponible en tu plan (p. ej. HTTP 402 por saldo
# de extra usage en cero), caemos automáticamente al fallback del plan gratuito.
OLLAMA_FALLBACK = "gpt-oss:120b-cloud"

from langchain_ollama import ChatOllama
from ollama import Client as OllamaClient

OLLAMA_HEADERS = {"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]}

def conectar_llm(**kwargs):
    # kwargs extra van directo a ChatOllama (p. ej. reasoning=True).
    probe = OllamaClient(host=OLLAMA_HOST, headers=OLLAMA_HEADERS)
    for modelo in [OLLAMA_MODEL, OLLAMA_FALLBACK]:
        try:
            probe.chat(model=modelo,
                       messages=[{"role": "user", "content": "ok"}],
                       options={"num_predict": 1})
        except Exception as exc:
            print(f"⚠ {modelo} no disponible: {str(exc)[:110]}")
            continue
        print("✔ Usando el modelo:", modelo)
        return ChatOllama(
            model=modelo,
            base_url=OLLAMA_HOST,
            client_kwargs={"headers": OLLAMA_HEADERS},
            temperature=0.1,
            **kwargs,
        )
    raise RuntimeError("Ningún modelo de Ollama cloud respondió; revisa OLLAMA_API_KEY.")

llm = conectar_llm()

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

cliente_mcp = MultiServerMCPClient({
    "tecnomarket": {
        "command": sys.executable,
        "args": [str(ROOT / "mcp" / "tecnomarket_server.py")],
        "transport": "stdio",
    },
})
tools_mcp = await cliente_mcp.get_tools()
print("Herramientas vía MCP:", [t.name for t in tools_mcp])

In [ ]:
# El mismo grafo del notebook 02, ahora sobre herramientas descubiertas por MCP.
from typing import Annotated, TypedDict

from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

class EstadoAgente(TypedDict):
    messages: Annotated[list, add_messages]

llm_con_tools = llm.bind_tools(tools_mcp)

async def nodo_agente(estado: EstadoAgente) -> dict:
    return {"messages": [await llm_con_tools.ainvoke(estado["messages"])]}

grafo = StateGraph(EstadoAgente)
grafo.add_node("agente", nodo_agente)
grafo.add_node("herramientas", ToolNode(tools_mcp))
grafo.add_edge(START, "agente")
grafo.add_conditional_edges("agente", tools_condition,
                            {"tools": "herramientas", END: END})
grafo.add_edge("herramientas", "agente")
agente = grafo.compile()
print("Agente con herramientas MCP compilado.")

In [ ]:
pregunta = ("Quiero regalar el reloj Aviator: dame su SKU y precio, y dime "
            "qué cubre su garantía si se daña la correa.")

async for paso in agente.astream({"messages": [HumanMessage(content=pregunta)]},
                                 stream_mode="values"):
    paso["messages"][-1].pretty_print()

## 4. El mismo servidor, en otros hosts

Ese `tecnomarket_server.py` que acabamos de consumir desde LangGraph funciona
sin cambios en cualquier host MCP. Por ejemplo, en **Claude Desktop / Claude
Code** bastaría registrar en la configuración:

```json
{
  "mcpServers": {
    "tecnomarket": {
      "command": "uv",
      "args": ["run", "--project", "/ruta/a/module4-genai",
               "python", "/ruta/a/module4-genai/mcp/tecnomarket_server.py"]
    }
  }
}
```

…y Claude podría buscar en el catálogo de TecnoMarket en medio de cualquier
conversación. Una herramienta, N hosts: esa es la promesa del protocolo.

## Resumen de la serie

| Pieza | Notebook | Rol |
|---|---|---|
| Qdrant + `gemini-embedding-2` | 01 | memoria vectorial multimodal |
| Chunking + híbrida + métricas | 01 | calidad de la recuperación |
| LangChain + Ollama cloud | 01 | generación fundamentada |
| LangGraph | 02 | el loop ReAct; RAG como herramienta |
| MCP (FastMCP + adapters) | 03 | las herramientas como servicio estándar |

**Ideas para extender**: transporte HTTP para servir el RAG a hosts remotos;
más tools (inventario, órdenes); orquestar la re-ingesta del índice con
Airflow (introducido en el Módulo 1); evaluar el agente de punta a punta
(¿elige la herramienta correcta? ¿cuántas llamadas gasta?).